# Baseline U-Net on LOL-v1 — first full training run (Colab GPU)

Protocol (fixed, do not change for this run):
- LOL-v1: train 450 / val 35 (from our485, split seed 42) / test eval15 (15)
- Baseline U-Net, L1 loss, crop 256, batch 4, Adam lr 2e-4, wd 0, 50 epochs, seed 42, no LR scheduler
- Best checkpoint selected by **validation PSNR only**
- Test set (eval15) evaluated **once**, after training, with the best checkpoint

**Before running:** Runtime → Change runtime type → **GPU**.

Expected Drive layout (upload from the Mac project, **without `.venv/`**):
```
MyDrive/LowLight-Generalization/
├── configs/  scripts/  src/  notebooks/  requirements.txt
└── data/LOL/LOLdataset.zip
```
Checkpoints, history and results are written inside this Drive folder (persistent).
The dataset is unzipped to the Colab local disk (`/content`) for fast reading; the zip on Drive is never modified.

## 1. Verify GPU (CUDA)

In [ ]:
import torch
print("torch", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
assert torch.cuda.is_available(), "No CUDA GPU. Runtime -> Change runtime type -> GPU, then restart."
print("CUDA device count:", torch.cuda.device_count())
print("Current CUDA device:", torch.cuda.current_device())
print("GPU name:", torch.cuda.get_device_name(0))
!nvidia-smi

## 2. Mount Google Drive and enter the project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/LowLight-Generalization'
for p in ['configs/baseline_lol_v1.yaml', 'scripts/train_baseline.py', 'src/models/baseline_unet.py',
          'data/LOL/LOLdataset.zip']:
    assert os.path.exists(os.path.join(PROJECT_DIR, p)), f'Missing on Drive: {p}'
%cd {PROJECT_DIR}
!ls

## 3. Environment (Colab already ships torch/torchvision/Pillow/numpy/matplotlib/PyYAML — torch is NOT reinstalled)

In [ ]:
import importlib
for m in ['torch', 'torchvision', 'PIL', 'numpy', 'matplotlib', 'yaml']:
    try:
        mod = importlib.import_module(m); print(f'{m:12s}', getattr(mod, '__version__', '?'))
    except ImportError:
        print(f'{m:12s} MISSING -> installing'); !pip -q install PyYAML

## 4. Copy LOL-v1 to local disk and verify the split (450 / 35 / 15)

In [ ]:
DATA_ROOT = '/content/LOL/LOLdataset'
if not os.path.isdir(os.path.join(DATA_ROOT, 'our485')):
    !mkdir -p {DATA_ROOT}
    !unzip -q {PROJECT_DIR}/data/LOL/LOLdataset.zip -d {DATA_ROOT} -x '__MACOSX/*'
!ls {DATA_ROOT}
!python scripts/check_lol_loader.py --root {DATA_ROOT}

## 5. Train (50 epochs)

Safety guard: refuses to start if a previous run already exists in `experiments/baseline_lol_v1/checkpoints/`,
so results are never silently overwritten. If Colab disconnects, set `RESUME = True` and re-run cells 1, 2, 4 and this one.

In [ ]:
RESUME = False
CKPT_DIR = os.path.join(PROJECT_DIR, 'experiments/baseline_lol_v1/checkpoints')
LOG_DIR = os.path.join(PROJECT_DIR, 'experiments/baseline_lol_v1')
os.makedirs(LOG_DIR, exist_ok=True)
last = os.path.join(CKPT_DIR, 'last.pth')

if RESUME:
    assert os.path.exists(last), 'Nothing to resume.'
    !python scripts/train_baseline.py --data-root {DATA_ROOT} --device cuda --resume {last} 2>&1 | tee -a {LOG_DIR}/train_log.txt
else:
    assert not os.path.exists(last), f'A previous run exists in {CKPT_DIR}. Move it away or set RESUME=True.'
    !python scripts/train_baseline.py --data-root {DATA_ROOT} --device cuda 2>&1 | tee {LOG_DIR}/train_log.txt

## 6. Check training finished (50 epochs, best.pth / last.pth / final.pth present)

In [ ]:
import json
s = json.load(open(os.path.join(LOG_DIR, 'train_summary.json')))
print(json.dumps(s, indent=2))
assert s['epochs_completed'] == 50, 'Training did not complete 50 epochs'
for f in ['best.pth', 'last.pth', 'final.pth']:
    assert os.path.exists(os.path.join(CKPT_DIR, f)), f'missing {f}'
print('OK')

## 7. Test evaluation — ONCE, best validation checkpoint, eval15

Guard: refuses to run again if test results already exist (the test set must not be used repeatedly).

In [ ]:
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results/baseline_lol_v1')
assert not os.path.exists(os.path.join(RESULTS_DIR, 'test_metrics.json')), 'Test set already evaluated for this run.'
!python scripts/evaluate_baseline.py --data-root {DATA_ROOT} --device cuda --per-image

## 8. Figures: test comparisons (input | output | reference) and training curves

In [ ]:
!python scripts/visualize_baseline_results.py --data-root {DATA_ROOT} --device cuda --n 4
!python scripts/visualize_baseline_results.py --data-root {DATA_ROOT} --device cuda --n 15 --out-name test_comparison_all15.png
!python scripts/plot_training_curves.py
from IPython.display import Image, display
display(Image(os.path.join(RESULTS_DIR, 'training_curves.png')))
display(Image(os.path.join(RESULTS_DIR, 'test_comparison.png')))

## 9. Final report (copy this output back)

In [ ]:
s = json.load(open(os.path.join(LOG_DIR, 'train_summary.json')))
t = json.load(open(os.path.join(RESULTS_DIR, 'test_metrics.json')))
report = {
    '1. GPU': s['gpu'],
    '2. Total training time (min)': round(s['train_time_s_sum_of_epochs'] / 60, 1),
    '3. Epochs completed': s['epochs_completed'],
    '4. Best validation epoch': s['best_epoch'],
    '5. Best validation PSNR (dB)': round(s['best_val_psnr'], 3),
    '6. Validation SSIM at best epoch': round(s['best_val_ssim'], 4),
    '7. Final training loss (L1)': round(s['final_train_loss'], 5),
    '8. Final validation loss (L1)': round(s['final_val_loss'], 5),
    '9. Test PSNR (dB)': round(t['mean_psnr'], 3),
    '10. Test SSIM': round(t['mean_ssim'], 4),
    '11. Best checkpoint': s['best_checkpoint'],
    '12. Training history': s['history'],
    '13. Test results': RESULTS_DIR,
    'test evaluated checkpoint epoch': t['epoch'],
}
for k, v in report.items():
    print(f'{k:35s} {v}')
print()
!grep -i -E "warn|error|traceback" {LOG_DIR}/train_log.txt || echo "14. No warnings/errors in train_log.txt"